<div style="background-color:#fdecef; padding:15px; border-radius:10px;">

<h2>🧹 Data Cleaning & 🛠 Feature Engineering for Netflix Data</h2>

<p>
This notebook focuses on preparing Netflix datasets for reliable analysis by performing
systematic data cleaning and feature engineering. The goal is to ensure data quality,
consistency, and usability before exploratory analysis and visualization.
</p>

<h4>🧼 Data Cleaning Tasks</h4>
<ul>
  <li>Handling missing values (NULL / NaN)</li>
  <li>Fixing incorrect or inconsistent data types (e.g., dates, numeric fields)</li>
  <li>Removing duplicate records to reduce redundancy</li>
  <li>Standardizing text fields such as titles, genres, and countries</li>
</ul>

<h4>🛠 Feature Engineering Tasks</h4>
<ul>
  <li>Creating new derived columns (e.g., content age, release decade)</li>
  <li>Categorizing continuous variables for better analysis</li>
  <li>Encoding and transforming features for downstream analysis</li>
</ul>

<p>
This notebook is part of a <b>multi-stage EDA pipeline</b>. All cleaning and feature engineering
steps are performed here, while detailed analysis and visualization are carried out in
the main EDA notebook. 📊
</p>

</div>


## 🔌⚙️ Connecting to MySQL using SQLAlchemy

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sqlalchemy import create_engine
from urllib.parse import quote_plus

db_user = 'root'
db_password = quote_plus("Mridul@123")   # @ → %40
db_host = '127.0.0.1'
db_port = 3306
db_name = 'github_db'

engine = create_engine(
    f"mysql+pymysql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
)


## 🎬 Titles Table : Raw Data Overview


In [3]:
#Load Titles Table
titles_raw=pd.read_sql(""" select * from netflix_titles""",engine)
titles_raw

,show_id,title,type,release_year,genre,rating,director,country,language,date_added,imdb_score,budget_millions,age_certification
0,464237c8-d,War forward personal,TV Show,2021,Comedy,TV-MA,James Carpenter,Japan,Spanish,2019-03-20,3.7,108.11,Kids
1,ef1d668f-9,Require hundred recognize,Movie,1981,Documentary,TV-MA,Cassandra Goodman,Japan,Japanese,2018-07-21,3.2,265.63,Kids
2,61e8d17c-4,Adult exactly tough,TV Show,2002,Comedy,G,Kathleen Rodriguez,Japan,Korean,2023-07-24,8.6,206.21,All
3,43e4d2cc-d,Arm war,Movie,1996,Horror,PG-13,Jessica Foster,United Kingdom,English,2022-01-17,7.6,228.62,Kids
4,050cb3d7-4,Beat view,Movie,1995,Comedy,TV-14,Kristin Ramirez,Canada,English,2023-07-23,4.7,272.07,All
...,...,...,...,...,...,...,...,...,...,...,...,...,...
19215,bd617de7-f,Miss may lawyer section,Movie,2001,Horror,PG,Erin Olson,Japan,French,2020-10-21,6.9,111.74,Adult
19216,fadd6027-4,At stay,Movie,1992,Action,PG-13,Elizabeth Mendez,Japan,Korean,2019-09-06,3.1,209.34,Kids
19217,f382b395-0,Why look term,Movie,2003,Drama,TV-MA,Melissa Terry,South Korea,Korean,2018-08-13,9.9,98.61,Kids
19218,b5e4f131-d,Letter take,TV Show,1994,Thriller,g,Taylor Suarez,United Kingdom,English,2019-04-29,6.9,274.71,Teen


## 🔍 Identified Data Quality Issues    

## 📌 Follow-Up Actions in Code  

-------------------------------------------------------------------------------------------------------------------------------

### 🧹  Find Duplicate Records


In [4]:
query="""
SELECT show_id,count(*) as Duplicates_Count FROM netflix_titles
group by show_id
having count(*)>1;
"""
duplicates=pd.read_sql(query,engine)
duplicates

,show_id,Duplicates_Count


In [5]:
duplicate_title_year = pd.read_sql("""
SELECT title, release_year, COUNT(*) as duplicate_count
FROM netflix_titles
GROUP BY title, release_year
HAVING COUNT(*) > 1
""", engine)

duplicate_title_year

,title,release_year,duplicate_count
0,,1995,6
1,Town,2021,2
2,,2025,7
3,,1986,6
4,Season,2001,2
5,,2019,2
6,,1992,3
7,,2000,3
8,,2016,6
9,Card,2005,2


In [6]:
titles_df = pd.read_sql("SELECT * FROM netflix_titles", engine)

duplicate_rows = titles_df[
    titles_df.duplicated(subset=["title", "release_year"], keep=False)
].sort_values(["title", "release_year"])

duplicate_rows


,show_id,title,type,release_year,genre,rating,director,country,language,date_added,imdb_score,budget_millions,age_certification
9179,ced3dfe7-6,,TV Show,1982,Comedy,TV-14,Aaron Coleman,Japan,French,2022-08-10,6.0,246.31,All
14778,4a327be3-6,,TV Show,1982,Comedy,TV-14,Diane Gonzalez,France,Korean,2018-07-29,3.3,157.29,Adult
18516,f2ff041c-5,,TV Show,1982,Thriller,tv-14,David Huynh,Canada,Hindi,2023-07-26,5.0,239.31,teen
2016,1a237031-9,,TV Show,1983,Thriller,TV-MA,Jesse Willis,South Korea,English,2018-05-08,2.1,41.28,Adult
2730,926e839f-6,,Movie,1983,Documentary,TV-MA,Brian Hall,Canada,Spanish,2020-10-26,5.2,172.39,Teen
...,...,...,...,...,...,...,...,...,...,...,...,...,...
507,adb87f64-7,Season,Movie,2001,Animation,G,Jacob Roberts,South Korea,Korean,2023-12-10,5.8,291.50,Teen
4556,fdeca735-1,Total,TV Show,1989,Horror,G,Amanda Preston,United Kingdom,English,2021-09-27,7.7,100.26,Adult
15584,24c91f2e-d,Total,Movie,1989,Horror,G,John Camacho,Japan,Korean,2019-02-25,6.4,105.08,Kids
243,86998cd6-2,Town,TV Show,2021,Sci-Fi,R,Michael Jensen,Canada,Japanese,2020-08-03,8.9,183.09,All


In [12]:
titles_clean.to_sql(
    name="netflix_titles_clean",
    con=engine,
    if_exists="replace",
    index=False
)


19076

## 📂 Load Clean Table

In [92]:
titles_df = pd.read_sql("SELECT * FROM netflix_titles_clean", engine)


## 🧹 Remove Null Values


In [98]:
titles_df.isnull().sum()


show_id              0
title                0
type                 0
release_year         0
genre                0
rating               0
director             0
country              0
language             0
date_added           0
imdb_score           0
budget_millions      0
age_certification    0
dtype: int64

In [97]:
titles_df=titles_df.dropna()
titles_df

,show_id,title,type,release_year,genre,rating,director,country,language,date_added,imdb_score,budget_millions,age_certification
0,464237c8-d,War forward personal,TV Show,2021,Comedy,TV-MA,James Carpenter,Japan,Spanish,2019-03-20,3.7,108.11,Kids
1,ef1d668f-9,Require hundred recognize,Movie,1981,Documentary,TV-MA,Cassandra Goodman,Japan,Japanese,2018-07-21,3.2,265.63,Kids
2,61e8d17c-4,Adult exactly tough,TV Show,2002,Comedy,G,Kathleen Rodriguez,Japan,Korean,2023-07-24,8.6,206.21,All
3,43e4d2cc-d,Arm war,Movie,1996,Horror,PG-13,Jessica Foster,United Kingdom,English,2022-01-17,7.6,228.62,Kids
4,050cb3d7-4,Beat view,Movie,1995,Comedy,TV-14,Kristin Ramirez,Canada,English,2023-07-23,4.7,272.07,All
...,...,...,...,...,...,...,...,...,...,...,...,...,...
18850,bd617de7-f,Miss may lawyer section,Movie,2001,Horror,PG,Erin Olson,Japan,French,2020-10-21,6.9,111.74,Adult
18851,fadd6027-4,At stay,Movie,1992,Action,PG-13,Elizabeth Mendez,Japan,Korean,2019-09-06,3.1,209.34,Kids
18852,f382b395-0,Why look term,Movie,2003,Drama,TV-MA,Melissa Terry,South Korea,Korean,2018-08-13,9.9,98.61,Kids
18853,b5e4f131-d,Letter take,TV Show,1994,Thriller,g,Taylor Suarez,United Kingdom,English,2019-04-29,6.9,274.71,Teen


## 🔍 Checking the Data Types

In [20]:
titles_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19076 entries, 0 to 19075
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   show_id            19076 non-null  object        
 1   title              19076 non-null  object        
 2   type               19076 non-null  object        
 3   release_year       19076 non-null  int64         
 4   genre              19076 non-null  object        
 5   rating             19076 non-null  object        
 6   director           19076 non-null  object        
 7   country            19076 non-null  object        
 8   language           19076 non-null  object        
 9   date_added         18855 non-null  datetime64[ns]
 10  imdb_score         19076 non-null  float64       
 11  budget_millions    19076 non-null  float64       
 12  age_certification  19076 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(9)
memory 

## 🕒 Convert to Datetime

In [19]:
titles_df["date_added"] = pd.to_datetime(titles_df["date_added"])


## ✂️ Removing Extra Spaces


In [21]:
titles_df["type"] = titles_df["type"].str.strip()
titles_df["rating"] = titles_df["rating"].str.strip()
titles_df["title"] = titles_df["title"].str.strip()


## 🏷️ Correcting Column Names


In [23]:
for col in titles_df.select_dtypes(include="object").columns:
    if titles_df[col].str.contains("3", na=False).any():
        print(f"Column with '3' issue: {col}")


Column with '3' issue: show_id
Column with '3' issue: title
Column with '3' issue: type
Column with '3' issue: genre
Column with '3' issue: rating
Column with '3' issue: director
Column with '3' issue: country
Column with '3' issue: language
Column with '3' issue: age_certification


In [24]:
cols_to_fix = ["title","type","genre", "director", "country","language","age_certification"]

for col in cols_to_fix:
    titles_df[col] = titles_df[col].str.replace("3", "e", regex=False)


## ⚠️ Checking for Null Values

In [27]:
titles_df.isnull().sum()


show_id                0
title                  0
type                   0
release_year           0
genre                  0
rating                 0
director               0
country                0
language               0
date_added           221
imdb_score             0
budget_millions        0
age_certification      0
dtype: int64

## 🗑️ Drop Missing Rows

In [28]:
titles_df = titles_df.dropna(subset=["date_added"])


In [85]:
titles_df.isnull().sum()

show_id              0
title                0
type                 0
release_year         0
genre                0
rating               0
director             0
country              0
language             0
date_added           0
imdb_score           0
budget_millions      0
age_certification    0
dtype: int64

In [99]:
titles_df.to_sql(
    "netflix_titles_clean",
    engine,
    if_exists="replace",
    index=False
)


17781

## ✅ Netflix Titles Table is Clean
---------------------------------------------------------------------------------------------------

## 🧼 Data Cleaning in `Netflix_Subscription` Table

In [100]:
query="""
Select * from netflix_subscriptions
"""
title_df=pd.read_sql(query,engine)
title_df

,subscription_id,show_id,subscription_type,subscription_price,billing_cycle,start_date
0,S1,464237c8-d,Standard,4.08,Yearly,2025-04-06
1,S2,ef1d668f-9,Basic,23.16,Monthly,2021-08-16
2,S3,61e8d17c-4,Standard,8.25,Yearly,2022-05-15
3,S4,43e4d2cc-d,Premium,16.06,Yearly,2023-04-20
4,S5,050cb3d7-4,Standard,12.48,Monthly,2025-11-17
...,...,...,...,...,...,...
19698,S19996,bd617de7-f,Premium,19.88,Yearly,2023-09-23
19699,S19997,fadd6027-4,Premium,11.59,Monthly,2022-12-15
19700,S19998,f382b395-0,Premium,15.20,Monthly,2025-10-30
19701,S19999,b5e4f131-d,Basic,15.28,Yearly,2021-10-06


## ⚠️ Check Missing Values

In [102]:
title_df.isnull().sum()

subscription_id       0
show_id               0
subscription_type     0
subscription_price    0
billing_cycle         0
start_date            0
dtype: int64

## 🔄 Handle Duplicates

In [38]:
title_df.duplicated().sum()

np.int64(0)

## 🔍 Checking Data Types



In [40]:
title_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19703 entries, 0 to 19702
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   subscription_id     19703 non-null  object 
 1   show_id             19703 non-null  object 
 2   subscription_type   19703 non-null  object 
 3   subscription_price  19703 non-null  float64
 4   billing_cycle       19703 non-null  object 
 5   start_date          19703 non-null  object 
dtypes: float64(1), object(5)
memory usage: 923.7+ KB


## 🗓️ Convert Start Date to Datetime

In [42]:
title_df["start_date"] = pd.to_datetime(title_df["start_date"], errors="coerce")


## ✂️ Removing Extra Spaces

In [49]:
title_df["subscription_type"] = title_df["subscription_type"].str.strip()
title_df["billing_cycle"] = title_df["billing_cycle"].str.strip()


## 🛠️ Correcting Data in Specific Columns

In [52]:
cols_to_fix = ["subscription_type","billing_cycle"]

for col in cols_to_fix:
    title_df[col] = title_df[col].str.replace("3", "e", regex=False)

## 🧹 Removing Null Values

In [104]:
title_df.isnull().sum()

subscription_id         0
show_id                 0
subscription_type       0
subscription_price      0
billing_cycle           0
start_date            256
dtype: int64

In [105]:
title_df=title_df.dropna()

In [103]:
query="""
SELECT * 
FROM netflix_subscriptions_clean
"""
title_df=pd.read_sql(query,engine)
title_df

,subscription_id,show_id,subscription_type,subscription_price,billing_cycle,start_date
0,S1,464237c8-d,Standard,4.08,Yearly,2025-04-06
1,S2,ef1d668f-9,Basic,23.16,Monthly,2021-08-16
2,S3,61e8d17c-4,Standard,8.25,Yearly,2022-05-15
3,S4,43e4d2cc-d,Premium,16.06,Yearly,2023-04-20
4,S5,050cb3d7-4,Standard,12.48,Monthly,2025-11-17
...,...,...,...,...,...,...
19698,S19996,bd617de7-f,Premium,19.88,Yearly,2023-09-23
19699,S19997,fadd6027-4,Premium,11.59,Monthly,2022-12-15
19700,S19998,f382b395-0,Premium,15.20,Monthly,2025-10-30
19701,S19999,b5e4f131-d,Basic,15.28,Yearly,2021-10-06


In [106]:
title_df.to_sql(
    "netflix_subscriptions_clean",
    engine,
    if_exists="replace",
    index=False
)

19447

## ✅ Netflix_Subscription Table is Clean
----------------------------------------------------------------------------------

## 🧹 Cleaning `Netflix_Reviews` Table

In [54]:
query="""
Select * from netflix_reviews
"""
reviews_df=pd.read_sql(query,engine)
reviews_df

,review_id,show_id,rating,sentiment,review_text,review_date
0,R1,464237c8-d,2.2,Positive,Product worry language everybody someone guess...,2022-05-16
1,R2,ef1d668f-9,4.8,Positive,May tonight throw itself team task performance...,2023-11-23
2,R3,61e8d17c-4,3.9,Negative,Bar lot argue notice clear certain line develo...,2025-03-10
3,R4,,2.3,Neutral,Keep sure mission modern population doctor siz...,2022-09-01
4,R5,050cb3d7-4,2.5,,Staff ability human society again many includi...,2022-09-01
...,...,...,...,...,...,...
19707,R19996,bd617de7-f,3.8,Neutral,Method mind responsibility bag central individ...,2024-01-16
19708,R19997,fadd6027-4,2.5,Negative,Bank always can dream drug easy attention movi...,2025-09-05
19709,R19998,f382b395-0,1.1,Neutral,Technology without take life set hospital whet...,2023-04-12
19710,R19999,b5e4f131-d,3.6,Negative,Appear shoulder attention stop participant tes...,2025-11-23


## ✂️ Standardizing Column Names

In [57]:
reviews_df.columns


Index(['review_id', 'show_id', 'rating', 'sentiment', 'review_text',
       'review_date'],
      dtype='object')

In [55]:
reviews_df.columns = (
    reviews_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)


In [56]:
reviews_df.columns


Index(['review_id', 'show_id', 'rating', 'sentiment', 'review_text',
       'review_date'],
      dtype='object')

## ⚠️ Checking for Missing Values

In [58]:
reviews_df.isna().sum()


review_id      0
show_id        0
rating         0
sentiment      0
review_text    0
review_date    0
dtype: int64

## 🔄 Checking for Duplicates

In [60]:
reviews_df.duplicated().sum()


np.int64(0)

## 🔍 Checking Data Types

In [61]:
reviews_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19712 entries, 0 to 19711
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   review_id    19712 non-null  object 
 1   show_id      19712 non-null  object 
 2   rating       19712 non-null  float64
 3   sentiment    19712 non-null  object 
 4   review_text  19712 non-null  object 
 5   review_date  19712 non-null  object 
dtypes: float64(1), object(5)
memory usage: 924.1+ KB


## 🗓️ Converting Columns to Datetime


In [62]:
reviews_df["review_date"] = pd.to_datetime(
    reviews_df["review_date"], errors="coerce"
)


In [63]:
reviews_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19712 entries, 0 to 19711
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   review_id    19712 non-null  object        
 1   show_id      19712 non-null  object        
 2   rating       19712 non-null  float64       
 3   sentiment    19712 non-null  object        
 4   review_text  19712 non-null  object        
 5   review_date  19436 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(1), object(4)
memory usage: 924.1+ KB


## 🏷️ Correcting Names

In [65]:
cols_to_fix = ["sentiment","review_text"]

for col in cols_to_fix:
    reviews_df[col] = reviews_df[col].str.replace("3", "e", regex=False)

In [82]:
reviews_df.to_sql(
    "netflix_reviews_clean",
    engine,
    if_exists="replace",
    index=False
)

19712

## ✅ Netflix_Reviews Table is Clean
---------------------------------------------------------------

## 🧹 Data Cleaning of `Netflix_Viewership` Table

In [66]:
query="""
select * from netflix_viewership
"""
viewership_df=pd.read_sql(query,engine)
viewership_df

,view_id,show_id,views_millions,watch_hours_millions,region,device_type,completion_rate
0,V1,464237c8-d,115.57,548.46,Europe,TV,79.96
1,V2,ef1d668f-9,,237.16,Europe,Laptop,71.75
2,V3,61e8d17c-4,194.33,53.11,South America,Laptop,46.23
3,V4,43e4d2cc-d,65.57,401.37,North America,Tablet,68.27
4,V5,050cb3d7-4,131.04,538.07,North America,Tablet,58.06
...,...,...,...,...,...,...,...
19445,V19996,bd617de7-f,133.33,559.52,South America,Tablet,71.79
19446,V19997,fadd6027-4,70.82,233.70,South America,Mobile,91.53
19447,V19998,f382b395-0,14.09,338.20,Asia,TV,83.68
19448,V19999,b5e4f131-d,57.49,119.40,Europe,TV,38.01


## ✂️ Checking for Extra Spaces in Column Names

In [67]:
viewership_df.columns

Index(['view_id', 'show_id', 'views_millions', 'watch_hours_millions',
       'region', 'device_type', 'completion_rate'],
      dtype='object')

## ⚠️ Checking for Missing Values

In [69]:
viewership_df.isna().sum()


view_id                 0
show_id                 0
views_millions          0
watch_hours_millions    0
region                  0
device_type             0
completion_rate         0
dtype: int64

## 🔄 Checking for Duplicate Rows

In [71]:
viewership_df.duplicated().sum()


np.int64(0)

In [73]:
viewership_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19450 entries, 0 to 19449
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   view_id               19450 non-null  object 
 1   show_id               19450 non-null  object 
 2   views_millions        19450 non-null  object 
 3   watch_hours_millions  19450 non-null  float64
 4   region                19450 non-null  object 
 5   device_type           19450 non-null  object 
 6   completion_rate       19450 non-null  float64
dtypes: float64(2), object(5)
memory usage: 1.0+ MB


In [75]:
viewership_df["views_millions"].unique()



array(['115.57', '', '194.33', ..., '16.62', '90.62', '186.77'],
      dtype=object)

In [76]:
import numpy as np

mask = pd.to_numeric(viewership_df["views_millions"], errors="coerce").isna()

viewership_df[mask]["views_millions"].head(10)


1      
124    
161    
244    
252    
261    
401    
419    
536    
608    
Name: views_millions, dtype: object

In [77]:
viewership_df["views_millions"] = (
    viewership_df["views_millions"]
    .astype(str)
    .str.strip()
    .replace("", None)
)


In [78]:
viewership_df["views_millions"] = pd.to_numeric(
    viewership_df["views_millions"],
    errors="coerce"
)


In [79]:
viewership_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19450 entries, 0 to 19449
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   view_id               19450 non-null  object 
 1   show_id               19450 non-null  object 
 2   views_millions        19190 non-null  float64
 3   watch_hours_millions  19450 non-null  float64
 4   region                19450 non-null  object 
 5   device_type           19450 non-null  object 
 6   completion_rate       19450 non-null  float64
dtypes: float64(3), object(4)
memory usage: 1.0+ MB


In [81]:
viewership_df.to_sql(
    "netflix_viewership_clean",
    engine,
    if_exists="replace",
    index=False
)

19450

## ✅ All Netflix Tables are Clean